# CAPCOM — train the Astro Pilot guide (Run all)

CAPCOM is the tiny chat guide that runs **inside the visitor's browser** next to the Astro Pilot demo: it answers from ≤ 3 retrieved
fact notes, then offers exactly one next step. This notebook runs the whole recipe on one Colab GPU and leaves a ready-to-serve web
model folder (ONNX int4 for transformers.js) on your Google Drive.

**Recipe** (design spec §6): S1 teacher SFT (LoRA on LFM2.5-1.2B-Instruct, merged) → S2 student SFT (full fine-tune of LFM2.5-350M)
→ S3 on-policy distillation (`trl.DistillationTrainer`, reverse KL) → S4 multi-turn-aware DPO (2-turn simulated futures, verifiable
scorer, `apo_zero`) → S5 optional GRPO (`dapo`, verifiable rewards) → S6 pick the best stage by eval (+ 0.9/0.1 merge with S3 if it
helps) → S7 export (int4: q4f16 for WebGPU + q4 for WebGPU without shader-f16, parity check) → `capcom-web-<RUN_NAME>.zip` on Drive.

## How to run
1. Upload **`capcom-data-v1.zip`** — on the LaCie drive at `/Volumes/LaCie/astro-pilot/chat/data/capcom-data-v1.zip` — to Google Drive
   as **`MyDrive/capcom/capcom-data-v1.zip`** (create the `capcom` folder).
2. *Runtime → Change runtime type →* a GPU: **A100** (best), L4 (fine), or T4 (free; lighter recipe: fp16, smaller batches, one DPO
   round, no GRPO).
3. *Runtime → Run all*, and approve the Google Drive pop-up. Optional: add a Colab secret `HF_TOKEN` (faster Hub downloads; needed only
   for `HF_PUSH`). The GPU cell prints the tier it detected and the hyper-parameters it chose for every stage.

**Resumable.** Every stage writes `DONE.json` to `MyDrive/capcom/runs/<RUN_NAME>/<stage>/` and is skipped on a re-run; a stage in
progress mirrors its newest checkpoint to Drive and resumes from it. After a disconnect: reconnect, *Run all* again.

## Expected time (full dataset; FLOP-based estimates, not measurements)
| GPU | S1 | S2 | S3 | S4 | S5 | evals + S6 + S7 | total | Colab units |
|---|---|---|---|---|---|---|---|---|
| A100-40 | 0.5 h | 0.5 h | 1 h | 1 h | 0.7 h | 0.5 h | ≈ 4–5 h | ≈ 25 |
| L4 | 1 h | 1 h | 1.5 h | 1.5 h | 1 h | 0.7 h | ≈ 6–7 h | ≈ 12 |
| T4 | 2 h | 2 h | 1.5 h | 1 h | off | 1 h | ≈ 7–8 h | ≈ 9 |

Outputs on Drive (`MyDrive/capcom/runs/<RUN_NAME>/`): every stage's model, `evals/eval_<stage>.json`, and
`capcom-web-<RUN_NAME>.zip` (the web model folder: `onnx/model_q4f16.onnx`, `onnx/model_q4.onnx`, tokenizer, `kb.json`, evals,
`capcom.json`). Copy the zip back to `/Volumes/LaCie/astro-pilot/chat/models/` for the browser bench.

## Configuration

In [ ]:
# ---- configuration: edit, then Runtime > Run all (CAPCOM_* environment variables override, for local tests) ----
import json, os
STUDENT = os.environ.get('CAPCOM_STUDENT', 'lfm350')          # 'lfm350' (primary) | 'smol135' (lite) | 'granite350' (Apache fallback)
TEACHER_SIZE = os.environ.get('CAPCOM_TEACHER_SIZE', '1.2b')  # '1.2b' | '2.6b' (>= 40 GB GPUs; LFM2-2.6B shares the student's vocabulary)
TEACHER_OVERRIDE = os.environ.get('CAPCOM_TEACHER') or None   # a Hub id / path replacing the teacher (smoke tests: the student itself)
STAGES = {'S1': True, 'S2': True, 'S3': True, 'S4': True, 'S5': 'auto', 'S6': True, 'S7': True}  # 'auto': GRPO off on T4
STAGES.update(json.loads(os.environ.get('CAPCOM_STAGES', '{}')))
SMOKE = int(os.environ.get('CAPCOM_SMOKE', '0'))              # 1 = a few steps per stage on tiny slices (a pipeline check, ~30 min)
DRIVE_DIR = os.environ.get('CAPCOM_DRIVE', '/content/drive/MyDrive/capcom')
DATA_ZIP = os.environ.get('CAPCOM_DATA_ZIP', 'capcom-data-v1.zip')
RUN_NAME = os.environ.get('CAPCOM_RUN', 'run1-smoke' if SMOKE else 'run1')
EXPORT_ACC = int(os.environ.get('CAPCOM_EXPORT_ACC', '4'))    # S7 int4 accuracy level: 4 = int8 activations (fast), 1 = float (A/B pending)
HF_PUSH = False                                               # push the web model folder to a private Hub repo (Colab secret HF_TOKEN)
HF_REPO = 'your-hf-name/capcom-' + STUDENT
LOCAL = os.environ.get('CAPCOM_LOCAL') == '1'                 # local run (chat/tests/nb_e2e.py): no Colab, no pip, paths under CAPCOM_ROOT
print(dict(STUDENT=STUDENT, TEACHER_SIZE=TEACHER_SIZE, TEACHER_OVERRIDE=TEACHER_OVERRIDE, STAGES=STAGES, SMOKE=SMOKE, RUN_NAME=RUN_NAME,
           EXPORT_ACC=EXPORT_ACC, LOCAL=LOCAL))

## Install

In [ ]:
# ---- install the pinned stack (never torch: Colab's preinstalled torch stays; a constraint makes pip fail rather than replace it) ----
import importlib.metadata as md, subprocess, sys
PINS = ['transformers==5.18.0', 'trl==1.14.1', 'peft==0.21.1', 'accelerate==1.15.0', 'datasets==5.0.1', 'huggingface-hub==1.33.0', 'onnx==1.23.1', 'onnxscript==0.7.2', 'onnxruntime==1.30.0', 'onnxruntime-genai==0.17.1']
if LOCAL: print('local run: pip skipped')
else:
    torch_v = md.version('torch'); open('/tmp/capcom-constraints.txt', 'w').write(f'torch=={torch_v}\n')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', '/tmp/capcom-constraints.txt', *PINS], check=True)
    assert md.version('torch') == torch_v, 'torch was changed by pip'
got = {p.split('==')[0]: md.version(p.split('==')[0]) for p in PINS}
print('python', sys.version.split()[0], '| torch', md.version('torch'), '|', ' '.join(f'{k} {v}' for k, v in got.items()))
bad = {p: got[p.split('==')[0]] for p in PINS if got[p.split('==')[0]] != p.split('==')[1]}
assert not bad, f'version mismatch: {bad}'
import torch, transformers, trl, peft, accelerate, datasets, onnx, onnxruntime, onnxruntime_genai  # noqa: import check

## Library
The CAPCOM modules, written from the repository by `chat/colab/make_notebook.py` (no need to read them).

In [ ]:
# ---- where the library, data and runs live ----
from pathlib import Path
ROOT = Path(os.environ['CAPCOM_ROOT']) if LOCAL else Path('/content/capcom')
CODE = ROOT / 'code'
for d in ('chat/data', 'chat/train'): (CODE / d).mkdir(parents=True, exist_ok=True)
for f in ('chat/__init__.py', 'chat/data/__init__.py'): (CODE / f).touch()  # empty package files (%%writefile refuses an empty body)
print('library ->', CODE)

In [ ]:
%%writefile {CODE}/chat/prompt.py
"""chat/prompt.py — CAPCOM's prompt contract v1 (spec 2026-09-30-astro-pilot-capcom-design.md §4), the same text as chat/web/prompt.js:
a fixed preamble, up to MAX_NOTES retrieved notes, a state line (scene + <= 3 not-yet-seen highlights in tour order), then the last
HISTORY messages (two exchanges) and the visitor's message. Training rows, the notebook, eval and the page all build prompts here."""

PREAMBLE = ('You are CAPCOM, the guide inside Astro Pilot, a spaceflight demo running in this browser. '
            'Answer briefly from the notes, then offer one next step.')
MAX_NOTES, HISTORY, MAX_UNSEEN = 3, 4, 3
# the demo's highlights in tour order: (key, the name the state line uses)
HIGHLIGHTS = (
    ('belt', 'the asteroid belt'), ('comets', 'comets'), ('orbit', 'Earth orbit'), ('atmo', 'atmospheric flight'),
    ('cities', 'city skylines'), ('alps', 'the Alps'), ('pillars', 'the Zhangjiajie pillars'), ('weather', 'storm weather'),
    ('landing', 'the airliner landing'), ('docking', 'station docking'), ('moon', 'lunar orbit'), ('zoom', 'the Earth-zoom telescope'),
    ('training', 'live training'), ('narrator', 'the Narrator'),
)
NAMES = dict(HIGHLIGHTS)

def unseen(scene, seen=()):
    """The first MAX_UNSEEN highlight names in tour order that are neither the current scene nor already seen (keys)."""
    skip = {scene, *seen}
    return [name for key, name in HIGHLIGHTS if key not in skip][:MAX_UNSEEN]

def state_line(state):
    """'State: <scene name>; not seen yet: a, b, c.' from {'scene': key, 'seen': [keys]} (None: no state line)."""
    if not state: return None
    left = unseen(state['scene'], state.get('seen', ()))
    tail = f"; not seen yet: {', '.join(left)}." if left else '; everything seen.'
    return f"State: {NAMES.get(state['scene'], state['scene'])}{tail}"

def system_text(notes, state=None):
    """notes: fact texts, all rendered (the retriever returns at most MAX_NOTES; training rows may show a 4th oracle fact)."""
    lines = [PREAMBLE]
    lines += ['Notes:', *[f'- {n}' for n in notes]] if notes else ['Notes: none.']
    s = state_line(state)
    if s: lines.append(s)
    return '\n'.join(lines)

def messages(history, user, notes, state=None):
    """[system, <= HISTORY previous messages, the user message]; history: [{'role','content'}] alternating, oldest first."""
    h = [{'role': m['role'], 'content': m['content']} for m in history][-HISTORY:]
    return [{'role': 'system', 'content': system_text(notes, state)}, *h, {'role': 'user', 'content': user}]

def prev_user(history):
    """The previous visitor message (the retriever's follow-up context), or ''."""
    return next((m['content'] for m in reversed(history) if m['role'] == 'user'), '')


In [ ]:
%%writefile {CODE}/chat/retrieval.py
"""chat/retrieval.py — CAPCOM's fact retriever: BM25 over the knowledge-base facts, the same arithmetic as chat/web/retriever.js
(the browser side), so training rows see exactly the facts the page will show. Tokens: lowercase ASCII words and numbers,
stop words dropped, plurals stemmed (plural()); a fact is indexed as title + text + keywords twice. The query is the visitor's
message plus the previous visitor message at weight PREV_W (follow-ups such as "and how fast is it?"). Ties break by id."""
import json, math, re
from pathlib import Path

K1, B, TOP_K, PREV_W, MIN_SCORE = 1.2, 0.75, 3, 0.35, 2.0
STOP = frozenset('''a an the is are was were be been being am of to in on for with and or but it its this that these those what which who whom
how why when where do does did done can could would should will shall may might must i you he she we they me my your our their them us as at
by from up about into than then so if not no yes please tell explain there here some any all just also very really much many more most too
hi hey hello thanks thank ok okay'''.split())
_WORD = re.compile(r"[a-z0-9]+")

def plural(w):
    """A light plural stemmer (the same rules as retriever.js): -ies -> y, -ches/-shes/-xes/-sses -> drop es, -s -> drop (not -ss/-us/-is)."""
    if len(w) > 4 and w.endswith('ies'): return w[:-3] + 'y'
    if len(w) > 4 and w.endswith(('ches', 'shes', 'xes', 'sses')): return w[:-2]
    if len(w) > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')): return w[:-1]
    return w

def tokens(text):
    out = []
    for w in _WORD.findall(text.lower()):
        if w in STOP: continue
        w = plural(w)  # the stop list is checked before and after stemming ('does' must not become 'doe')
        if w not in STOP: out.append(w)
    return out

def doc_tokens(fact):
    kw = ' '.join(fact.get('keywords', []))
    return tokens(f"{fact.get('title', '')} {fact['text']} {kw} {kw}")

class BM25:
    def __init__(self, facts):
        self.facts = list(facts); self.ids = [f['id'] for f in self.facts]; self.by_id = {f['id']: f for f in self.facts}
        self.tf, self.len, df = [], [], {}
        for f in self.facts:
            t = doc_tokens(f); c = {}
            for w in t: c[w] = c.get(w, 0) + 1
            self.tf.append(c); self.len.append(len(t))
            for w in c: df[w] = df.get(w, 0) + 1
        n = len(self.facts); self.avg = sum(self.len) / max(n, 1)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}

    def query(self, text, prev=''):
        """{token: weight}, summed in first-seen order (JS iterates the same Map order)."""
        q = {}
        for w in tokens(text): q[w] = q.get(w, 0.0) + 1.0
        for w in tokens(prev): q[w] = q.get(w, 0.0) + PREV_W
        return q

    def scores(self, text, prev=''):
        q, out = self.query(text, prev), []
        for i, c in enumerate(self.tf):
            s, norm = 0.0, K1 * (1 - B + B * self.len[i] / self.avg)
            for w, qw in q.items():
                f = c.get(w)
                if f: s += qw * self.idf[w] * f * (K1 + 1) / (f + norm)
            if s > 0: out.append((s, self.ids[i]))
        out.sort(key=lambda x: (-x[0], x[1]))
        return out

    def search(self, text, prev='', k=TOP_K, min_score=MIN_SCORE):
        """[(id, score)] of the top k facts scoring at least min_score."""
        return [(i, s) for s, i in self.scores(text, prev)[:k] if s >= min_score]

def load_facts(path):
    p = Path(path)
    if p.suffix == '.json': return json.loads(p.read_text())['facts']
    return [json.loads(l) for l in p.read_text().split('\n') if l.strip()]


In [ ]:
%%writefile {CODE}/chat/rewards.py
"""chat/rewards.py — CAPCOM's verifiable reply scorer (spec R6): one function used by eval, DPO pair ranking and GRPO rewards.
score(answer, row) -> {'total': 0..1, <component>: 0..1 or None (not applicable), ...}. row (a prompt row from chat/data/build.py):
  notes [str] shown to the model, user str, must [str] key terms (retrieval tokens) the reference answer grounded in the notes,
  abstain bool (the notes lack the answer: an honest "not in my notes" + redirect is right), lead bool (a lead move is expected).
Components: recall (share of must terms), grounded (numbers / ?flags / keys all appear in the notes or the visitor's message),
lead (exactly one lead move at the end; two or more questions = nagging), first (does not open with a question when it has more to
say), length (8-70 words), abstain (abstains iff it should), repetition (repeated 4-grams), leak (no role/prompt text)."""
import re
from chat.retrieval import tokens

W = {'recall': 0.30, 'grounded': 0.20, 'lead': 0.15, 'abstain': 0.10, 'length': 0.10, 'first': 0.05, 'repetition': 0.05, 'leak': 0.05}
MIN_W, MAX_W = 8, 70
NUM = re.compile(r'(?<![\w.])\d[\d,]*(?:\.\d+)?')
FLAG = re.compile(r'\?[a-z][a-z_]*(?:=[\w.,-]+)?')
KEY = re.compile(r"\b(?:press|key|hit|tap)\s+['\"]?([A-Za-z0-9+\-])['\"]?(?![\w-])", re.I)
ABSTAIN = re.compile(r"\b(not in my (?:flight )?notes|(?:i )?don'?t (?:know|have)|not sure|no (?:info|information|notes)|can'?t (?:tell|say|find)|"
                     r"isn'?t (?:covered|something i)|outside (?:what|my)|beyond (?:what|my)|i'?m (?:only|just) (?:the|a) guide|not (?:something )?i can help)", re.I)
CUE = re.compile(r"\b(try|open|switch|press|click|tap|want to|would you like|how about|check out|take a look|head (?:to|over)|flip|turn on|"
                 r"toggle|pick|jump|watch|ask me|curious|shall we|next)\b", re.I)
LEAK = re.compile(r"<\|im_|<\|endoftext|^(?:user|assistant|system|capcom)\s*:|\n(?:user|assistant|system)\s*:|^notes:|\nnotes:|^state:|\nstate:", re.I)
SMALL = {str(i) for i in range(11)}
QMARK = re.compile(r'\?(?![A-Za-z_])')  # a question mark, not the '?' that starts a URL flag such as ?scenario=landing

def questions(text): return len(QMARK.findall(text))

def norm_num(s):
    s = s.replace(',', '')
    return s[:-2] if s.endswith('.0') else s

def sentences(text):
    return [s for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s]

def comp_recall(answer, must):
    if not must: return None
    have = set(tokens(answer))
    return sum(t in have for t in must) / len(must)

def comp_grounded(answer, notes, user):
    src = ' '.join(notes) + ' ' + user
    nums = {norm_num(n) for n in NUM.findall(src)}
    bad = [n for n in (norm_num(x) for x in NUM.findall(answer)) if n not in nums and n not in SMALL]
    low = src.lower()
    bad += [f for f in FLAG.findall(answer.lower()) if f not in low]
    # a key is grounded when that exact character stands alone somewhere in the notes/message ("key Z", "(Z)", "Z opens…")
    bad += [k for k in KEY.findall(answer) if not re.search(rf'(?<![A-Za-z0-9]){re.escape(k)}(?![A-Za-z0-9])', src)]
    return max(0.0, 1.0 - 0.5 * len(bad))

def comp_lead(answer, expected=True):
    q = questions(answer); s = sentences(answer); last = s[-1] if s else ''
    if not expected: return 1.0 if q == 0 else 0.5
    if q >= 2: return 0.0
    return 1.0 if QMARK.search(last) or CUE.search(last) else 0.0

def comp_first(answer):
    s = sentences(answer)
    return 0.0 if len(s) >= 2 and s[0].endswith('?') else 1.0

def comp_length(answer):
    n = len(answer.split())
    if n < MIN_W: return n / MIN_W
    return 1.0 if n <= MAX_W else max(0.0, 1.0 - (n - MAX_W) / 60)

def comp_abstain(answer, abstain, must):
    said = bool(ABSTAIN.search(answer))
    if abstain: return 1.0 if said else 0.0
    return 0.5 if said and must else None

def comp_repetition(answer):
    w = answer.lower().split(); grams = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    if not grams: return 1.0
    return max(0.0, 1.0 - 3 * (1 - len(set(grams)) / len(grams)))

def comp_leak(answer):
    return 0.0 if LEAK.search(answer.strip()) else 1.0

def score(answer, row):
    answer = (answer or '').strip(); must = row.get('must') or []
    c = {'recall': None if row.get('abstain') else comp_recall(answer, must),
         'grounded': comp_grounded(answer, row.get('notes', []), row.get('user', '')),
         'lead': comp_lead(answer, row.get('lead', True)), 'first': comp_first(answer), 'length': comp_length(answer),
         'abstain': comp_abstain(answer, bool(row.get('abstain')), must), 'repetition': comp_repetition(answer), 'leak': comp_leak(answer)}
    if not answer: c = {k: (0.0 if v is not None else None) for k, v in c.items()}
    w = [k for k, v in c.items() if v is not None]
    c['total'] = sum(W[k] * c[k] for k in w) / sum(W[k] for k in w)
    return c


In [ ]:
%%writefile {CODE}/chat/export_web.py
"""chat/export_web.py — export a (fine-tuned) causal LM to the folder transformers.js 4.3 loads (spec R7): the ONNX Runtime GenAI model
builder's fused int4 graphs (GroupQueryAttention, SkipSimplifiedLayerNorm, MatMulNBits block 32, 4-bit tied embeddings via
GatherBlockQuantized, LFM2's CausalConvWithState) and fix_io() for the transformers.js contract:
- LFM2 conv states past.N.conv / present.N.conv -> past_conv.N / present_conv.N (transformers.js getCacheNames);
- the symbolic kv_cache_dim -> head_dim (transformers.js sizes the empty cache from the input metadata; unknown symbols become 0);
- a scalar int64 input num_logits_to_keep that slices the hidden states before the LM head, so a prefill computes one row of logits
  instead of prompt_len x vocab (transformers.js feeds it whenever the graph declares it);
- one file, no external data.
max_position_embeddings is cut to --ctx first, so the rotary cos/sin caches hold ctx rows, not 128k.
dtypes: q4f16 = int4 weights, fp16 activations/IO, built for WebGPU; q4 = int4 weights, fp32 IO, built for CPU/WASM (also runs on WebGPU).
  python -m chat.export_web --src <hf dir> --out <folder> [--dtypes q4f16,q4] [--ctx 4096] [--check]"""
import argparse, json, os, shutil, subprocess, sys, tempfile
from pathlib import Path
import onnx
from onnx import TensorProto, helper

SUFFIX = {'q4f16': '_q4f16', 'q4': '_q4', 'fp16': '_fp16'}
# int4 accuracy level: 4 lets MatMulNBits quantize activations to int8 (fast; on WebGPU it costs quality), 1 keeps them in float
BUILD = {'q4f16': ('int4', 'webgpu', ['int4_block_size=32']), 'q4': ('int4', 'cpu', ['int4_block_size=32']), 'fp16': ('fp16', 'webgpu', [])}
ACC = {'q4f16': 4, 'q4': 4}
TOKENIZER_FILES = ('tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt')

def rename(g, old, new):
    for n in g.node:
        n.input[:] = [new if i == old else i for i in n.input]; n.output[:] = [new if o == old else o for o in n.output]
    for v in [*g.input, *g.output]:
        if v.name == old: v.name = new

def add_num_logits_to_keep(g):
    """Insert Slice(hidden, -k, INT64_MAX, axis 1) in front of the LM head (the MatMul/MatMulNBits that produces the logits)."""
    prod = {o: n for n in g.node for o in n.output}
    head = prod['logits']
    while head.op_type in ('Cast', 'Identity'): head = prod[head.input[0]]
    if head.op_type not in ('MatMul', 'MatMulNBits'): raise ValueError(f'unexpected LM head op {head.op_type}')
    src = head.input[0]
    g.input.append(helper.make_tensor_value_info('num_logits_to_keep', TensorProto.INT64, []))
    g.initializer.extend([helper.make_tensor('nltk/axes', TensorProto.INT64, [1], [1]), helper.make_tensor('nltk/zero', TensorProto.INT64, [1], [0]),
                          helper.make_tensor('nltk/end', TensorProto.INT64, [1], [2 ** 62])])
    nodes = [helper.make_node('Neg', ['num_logits_to_keep'], ['nltk/neg']),
             helper.make_node('Unsqueeze', ['nltk/neg', 'nltk/zero'], ['nltk/start']),
             helper.make_node('Slice', [src, 'nltk/start', 'nltk/end', 'nltk/axes'], ['nltk/hidden'])]
    head.input[0] = 'nltk/hidden'
    i = list(g.node).index(head)
    for k, n in enumerate(nodes): g.node.insert(i + k, n)
    for v in g.output:
        if v.name == 'logits': v.type.tensor_type.shape.dim[1].dim_param = 'num_logits_to_keep'

def fix_io(src, dst, head_dim):
    m = onnx.load(str(src)); g = m.graph
    for v in list(g.input) + list(g.output):
        p = v.name.split('.')
        if len(p) == 3 and p[0] in ('past', 'present') and p[2] == 'conv': rename(g, v.name, f'{p[0]}_conv.{p[1]}')
    for v in [*g.input, *g.output]:
        if v.name.startswith(('past_key_values.', 'present.')):
            d = v.type.tensor_type.shape.dim[3]; d.Clear(); d.dim_value = head_dim
    add_num_logits_to_keep(g)
    onnx.save(m, str(dst), save_as_external_data=False)

def prepare_src(src, work, ctx):
    """A copy of the HF folder whose config caps max_position_embeddings at ctx (smaller rotary caches)."""
    d = Path(work) / 'src'
    shutil.copytree(src, d, ignore=shutil.ignore_patterns('._*', 'optimizer.pt', 'rng_state*', 'scheduler.pt', 'trainer_state.json', 'training_args.bin'))
    cfg = json.loads((d / 'config.json').read_text())
    for k in ('max_position_embeddings', 'seq_length'):
        if k in cfg: cfg[k] = min(int(cfg[k]), ctx)
    (d / 'config.json').write_text(json.dumps(cfg, indent=1))
    return d, cfg

def build(src_dir, cfg, dtype, work, acc=None):
    precision, ep, extra = BUILD[dtype]; out = Path(work) / dtype
    if dtype in ACC: extra = [*extra, f'int4_accuracy_level={acc or ACC[dtype]}']
    shared = ['shared_embeddings=true'] if cfg.get('tie_word_embeddings', cfg.get('tie_embedding', False)) else []
    cmd = [sys.executable, '-m', 'onnxruntime_genai.models.builder', '-i', str(src_dir), '-o', str(out), '-p', precision, '-e', ep,
           '-c', str(Path(work) / 'cache'), '--extra_options', *extra, *shared]
    # the source is a local folder: offline, so the builder never blocks on (rate-limited, unauthenticated) Hub requests
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, env={**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'})
    return out / 'model.onnx'

def export(src, out, dtypes=('q4f16', 'q4'), ctx=4096, acc=None):
    out = Path(out); (out / 'onnx').mkdir(parents=True, exist_ok=True)
    work = Path(tempfile.mkdtemp(prefix='.export_', dir=out.parent))
    try:
        s, cfg = prepare_src(src, work, ctx)
        hd = cfg.get('head_dim') or cfg['hidden_size'] // cfg['num_attention_heads']
        for d in dtypes: fix_io(build(s, cfg, d, work, acc), out / 'onnx' / f'model{SUFFIX[d]}.onnx', hd)
        for f in TOKENIZER_FILES + ('config.json', 'generation_config.json', 'chat_template.jinja'):
            if (s / f).exists(): shutil.copy(s / f, out / f)
        tk = out / 'tokenizer_config.json'
        if (s / 'chat_template.jinja').exists() and tk.exists():
            t = json.loads(tk.read_text()); t['chat_template'] = (s / 'chat_template.jinja').read_text(); tk.write_text(json.dumps(t, indent=1))
    finally:
        shutil.rmtree(work, ignore_errors=True)
    return {d: round((out / 'onnx' / f'model{SUFFIX[d]}.onnx').stat().st_size / 2 ** 20, 1) for d in dtypes}

def check(src, out, prompts=('Hi! What is Astro Pilot?', 'How does the pilot learn to dodge asteroids?'), n=24):
    """Greedy parity on CPU: PyTorch fp32 vs the q4 graph (ORT CPU) — the share of identical next tokens along the torch path."""
    import numpy as np, onnxruntime as ort, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(src); model = AutoModelForCausalLM.from_pretrained(src, dtype=torch.float32).eval()
    sess = ort.InferenceSession(str(Path(out) / 'onnx' / 'model_q4.onnx'), providers=['CPUExecutionProvider'])
    meta = {i.name: i for i in sess.get_inputs()}; agree = total = 0
    for p in prompts:
        ids = tok.apply_chat_template([{'role': 'user', 'content': p}], add_generation_prompt=True, return_tensors='pt')
        ids = ids['input_ids'] if hasattr(ids, 'keys') else ids
        with torch.no_grad(): ref = model.generate(ids, max_new_tokens=n, do_sample=False)[0, ids.shape[1]:].tolist()
        seq = ids[0].tolist()
        for t in ref:
            feed = {'input_ids': np.array([seq], np.int64), 'attention_mask': np.ones((1, len(seq)), np.int64), 'num_logits_to_keep': np.array(1, np.int64)}
            for k, v in meta.items():
                if k in feed: continue
                shape = [1 if d == 'batch_size' else 0 if isinstance(d, str) else d for d in v.shape]
                feed[k] = np.zeros(shape, np.float16 if 'float16' in v.type else np.float32)
            logits = sess.run(['logits'], feed)[0]
            agree += int(logits[0, -1].argmax() == t); total += 1; seq.append(t)
    return {'greedy_agreement': agree / total, 'tokens': total}

if __name__ == '__main__':
    os.environ.setdefault('HF_HOME', '/Volumes/LaCie/astro-pilot/chat/hf')
    ap = argparse.ArgumentParser(); ap.add_argument('--src', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--dtypes', default='q4f16,q4'); ap.add_argument('--ctx', type=int, default=4096); ap.add_argument('--check', action='store_true')
    ap.add_argument('--acc', type=int, choices=(1, 2, 3, 4), help='int4 accuracy level (default 4)')
    a = ap.parse_args(); sizes = export(a.src, a.out, a.dtypes.split(','), a.ctx, a.acc); print(json.dumps({'sizes_mb': sizes}))
    if a.check: print(json.dumps(check(a.src, a.out)))


In [ ]:
%%writefile {CODE}/chat/train/__init__.py
"""chat/train — CAPCOM's training stages (spec §6): S1 teacher SFT, S2 student SFT (sft_stage), S3 on-policy distillation
(distill_stage), S4 multi-turn-aware DPO (dpo_stage), S5 optional GRPO (grpo_stage), S6 pick best + merge (merge_stage), S7 web
export (web_stage). Shared plumbing (GPU tier, models, hyper-parameters, data, DONE markers, Drive mirroring) lives in common."""


In [ ]:
%%writefile {CODE}/chat/train/common.py
"""chat/train/common.py — shared plumbing for the CAPCOM stages (spec §6; rulings R1/R2/R5): GPU probe -> tier and precision (T4 has
no bf16: fp16), the model registry, the stage x tier hyper-parameter table (+ SMOKE overrides: a few steps on tiny slices), data loading
from the package (SFT rows mixed with general replay, prompt rows), notes retrieval (the page's BM25 + min_score), the visitor simulator
prompt, batched chat generation, and Run: a local work dir mirrored to Drive with per-stage DONE markers and checkpoint resume."""
import gc, hashlib, json, os, random, shutil, time
from pathlib import Path
from chat.prompt import MAX_NOTES, prev_user
from chat.retrieval import BM25, MIN_SCORE

GENERAL_FRAC = 0.25        # share of general-replay rows in the SFT mix (spec §5)
DEFAULT_STATE = {'scene': 'belt', 'seen': []}
MODELS = {  # R1/R2: the student, its same-vocabulary teacher, the >= 40 GB teacher; s2_lr = the student's full fine-tune lr
    'lfm350': {'student': 'LiquidAI/LFM2.5-350M', 'teacher': 'LiquidAI/LFM2.5-1.2B-Instruct',
               # LFM2.5-2.6B has a 128k vocabulary; DistillationTrainer needs the student's 65,536 — LFM2-2.6B shares it (ids 0-395 and
               # every BPE/chat token are identical; only reserved image slots are named differently)
               'teacher_big': 'LiquidAI/LFM2-2.6B', 's2_lr': 3e-5},
    'smol135': {'student': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'teacher': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
                'teacher_big': 'HuggingFaceTB/SmolLM2-1.7B-Instruct', 's2_lr': 5e-5},
    'granite350': {'student': 'ibm-granite/granite-4.0-350m', 'teacher': 'ibm-granite/granite-4.0-1b',
                   'teacher_big': 'ibm-granite/granite-4.0-1b', 's2_lr': 3e-5},
}
# hyper-parameters: A100-40 defaults (spec §6 table), per-tier overrides, SMOKE overrides. bs = per-device batch, ga = accumulation.
BASE = {
    's1': dict(lr=1e-4, epochs=2, bs=16, ga=4, max_len=1024, lora_r=64, lora_alpha=128, max_rows=None, max_steps=-1, save=0.25),
    's2': dict(lr=None, epochs=3, bs=32, ga=2, max_len=1024, warmup=0.03, max_rows=None, max_steps=-1, save=0.25),
    's3': dict(lr=1e-5, max_steps=400, bs=16, ga=2, max_completion=128, temperature=1.0, beta=1.0, n_prompts=8000, save=0.25),
    's4': dict(n_ctx=2000, k=4, temperature=0.8, max_new=128, sim_tokens=48, gen_bs=64, rounds=2, margin=0.15,
               lr=1e-6, epochs=2, bs=8, ga=4, beta=0.3, max_len=1024, max_steps=-1, save=0),
    's5': dict(lr=1e-6, max_steps=200, bs=16, ga=2, num_generations=4, beta=0.02, epsilon_high=0.28, max_completion=128,
               n_prompts=4000, save=0.25),
    'eval': dict(n_single=None, n_unans=None, n_val=200, n_scripts=None, turns=4, bs=64, max_new=120),
}
_H100 = {'s1': dict(bs=32, ga=2), 's2': dict(bs=64, ga=1), 's3': dict(max_steps=500, bs=32, ga=1),
         's4': dict(n_ctx=3000, gen_bs=128, bs=16, ga=2), 's5': dict(bs=32, ga=1), 'eval': dict(bs=128)}
TIER = {
    't4': {'s1': dict(epochs=1, bs=4, ga=16, max_rows=20000), 's2': dict(epochs=2, bs=8, ga=8),
           's3': dict(max_steps=200, bs=8, ga=4, n_prompts=6400), 's4': dict(n_ctx=600, gen_bs=16, rounds=1, bs=2, ga=16),
           's5': dict(num_generations=2, bs=4, ga=8, max_steps=150), 'eval': dict(bs=16)},
    'l4': {'s1': dict(epochs=1, bs=8, ga=8), 's2': dict(bs=16, ga=4), 's3': dict(max_steps=300, bs=8, ga=4),
           's4': dict(n_ctx=1200, gen_bs=32, bs=4, ga=8), 's5': dict(bs=8, ga=4), 'eval': dict(bs=32)},
    'a100': {}, 'h100': _H100, 'big': _H100,
    'mps': {'s1': dict(epochs=1, bs=2, ga=32, max_rows=4000), 's2': dict(epochs=1, bs=4, ga=16, max_rows=8000),
            's3': dict(max_steps=50, bs=4, ga=4, n_prompts=800), 's4': dict(n_ctx=100, gen_bs=16, rounds=1, bs=2, ga=16),
            's5': dict(bs=4, ga=4, num_generations=2, max_steps=30), 'eval': dict(bs=16, n_val=100)},
    'cpu': {'s1': dict(epochs=1, bs=1, ga=16, max_rows=500), 's2': dict(epochs=1, bs=2, ga=16, max_rows=1000),
            's3': dict(max_steps=10, bs=2, ga=4, n_prompts=80), 's4': dict(n_ctx=20, gen_bs=8, rounds=1, bs=1, ga=8),
            's5': dict(bs=2, ga=4, num_generations=2, max_steps=10), 'eval': dict(bs=8, n_val=40)},
}
SMOKE = {
    's1': dict(max_steps=3, bs=2, ga=1, max_rows=24, lora_r=8, lora_alpha=16, max_len=768, save=2),
    's2': dict(max_steps=4, bs=2, ga=1, max_rows=32, max_len=768, save=0),
    's3': dict(max_steps=2, bs=2, ga=1, max_completion=32, n_prompts=8, save=0),
    's4': dict(n_ctx=3, k=4, max_new=32, sim_tokens=24, gen_bs=12, rounds=2, margin=0.0, epochs=1, bs=1, ga=1, max_steps=2, max_len=768),
    's5': dict(max_steps=2, bs=2, ga=1, num_generations=2, max_completion=32, n_prompts=8, save=0),
    'eval': dict(n_single=3, n_unans=3, n_val=4, n_scripts=2, turns=2, bs=8, max_new=48),
}

def hp(stage, tier, smoke=False, student='lfm350'):
    """The hyper-parameters of stage ('s1'..'s5', 'eval') on a tier: BASE <- TIER <- SMOKE."""
    h = {**BASE[stage], **TIER.get(tier, {}).get(stage, {})}
    if stage == 's2' and h['lr'] is None: h['lr'] = MODELS[student]['s2_lr']
    if smoke: h.update(SMOKE[stage])
    return h

def probe():
    """{'tier', 'device', 'name', 'mem_gb', 'cc', 'bf16', 'fp16'}: T4 -> fp16 (compute capability 7.5 has no bf16), Ampere+ -> bf16,
    MPS/CPU -> fp32. Tiers: t4, l4, a100 (40 GB), h100, big (other >= 80 GB cards), mps, cpu."""
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0); name, mem, cc = p.name, p.total_memory / 2 ** 30, (p.major, p.minor); n = name.upper()
        tier = ('t4' if 'T4' in n else 'l4' if 'L4' in n else 'h100' if ('H100' in n or 'H200' in n) else 'big' if mem >= 79
                else 'a100' if ('A100' in n or mem >= 38) else 'l4' if mem >= 22 else 't4')
        bf16 = cc >= (8, 0)
        return {'tier': tier, 'device': 'cuda', 'name': name, 'mem_gb': round(mem, 1), 'cc': f'{cc[0]}.{cc[1]}', 'bf16': bf16, 'fp16': not bf16}
    dev = 'mps' if torch.backends.mps.is_available() else 'cpu'
    return {'tier': dev, 'device': dev, 'name': dev, 'mem_gb': None, 'cc': None, 'bf16': False, 'fp16': False}

def default_stages(tier):
    """Stage toggles: everything on, S5 GRPO off on T4 (spec §6)."""
    return {'S1': True, 'S2': True, 'S3': True, 'S4': True, 'S5': tier != 't4', 'S6': True, 'S7': True}

def log(*a):
    print(time.strftime('[%H:%M:%S]'), *a, flush=True)

def seed_all(seed):
    import numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def free():
    import torch
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    elif torch.backends.mps.is_available(): torch.mps.empty_cache()

# ---- data ---------------------------------------------------------------------------------------------------------------------------
def read_jsonl(p):
    """Rows of a JSONL file; [] when it is missing or empty (the dev package has empty eval files)."""
    p = Path(p)  # split on '\n' only: str.splitlines() also breaks on U+2028/U+2029 inside JSON strings
    return [json.loads(l) for l in p.read_text(encoding='utf-8').split('\n') if l.strip()] if p.exists() else []

def find_package(d):
    """The dataset package folder (the one holding MANIFEST.json): d itself or its single child."""
    d = Path(d)
    if (d / 'MANIFEST.json').exists(): return d
    c = [p.parent for p in sorted(d.glob('*/MANIFEST.json')) if not p.parent.name.startswith(('.', '_'))]
    if len(c) != 1: raise FileNotFoundError(f'expected one package with MANIFEST.json under {d}, found {len(c)}')
    return c[0]

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def verify_manifest(pkg):
    """[(file, problem)] for every MANIFEST file that is missing or whose sha256 differs ([] = the package is intact)."""
    pkg = Path(pkg); man = json.loads((pkg / 'MANIFEST.json').read_text()); bad = []
    for f, m in man['files'].items():
        p = pkg / f
        if not p.exists(): bad.append((f, 'missing'))
        elif sha256(p) != m['sha256']: bad.append((f, 'sha256'))
    return bad

def mix_general(capcom, general, frac=GENERAL_FRAC, seed=0):
    """capcom + a random sample of general rows so that general is `frac` of the result (capped by what exists), shuffled."""
    n = min(len(general), round(frac * len(capcom) / (1 - frac))) if 0 < frac < 1 else 0
    rng = random.Random(seed); rows = capcom + (rng.sample(general, n) if n else [])
    rng.shuffle(rows)
    return rows

def pc(r):
    return {'prompt': r['prompt'], 'completion': r['completion']}

def sft_rows(pkg, split='train', general_frac=GENERAL_FRAC, max_rows=None, seed=0):
    """Prompt-completion rows: sft/capcom_<split> (+ sft/general replay on train), shuffled, cut to max_rows."""
    pkg = Path(pkg); capcom = [pc(r) for r in read_jsonl(pkg / 'sft' / f'capcom_{split}.jsonl')]
    general = [pc(r) for r in read_jsonl(pkg / 'sft' / 'general.jsonl')] if split == 'train' else []
    rows = mix_general(capcom, general, general_frac, seed)
    return rows[:max_rows] if max_rows else rows

def prompt_rows(pkg, split='train', n=None, seed=0):
    """Prompt rows with the scorer fields (prompts/<split>.jsonl), a seeded random n of them."""
    rows = read_jsonl(Path(pkg) / 'prompts' / f'{split}.jsonl')
    return random.Random(seed).sample(rows, n) if n and len(rows) > n else rows

class Notes:
    """The page's retrieval: BM25 top MAX_NOTES over kb.json facts, kb['retrieval']['min_score'] as the no-facts threshold."""
    def __init__(self, kb):
        self.bm = BM25(kb['facts']); self.min_score = kb.get('retrieval', {}).get('min_score', MIN_SCORE)
    def __call__(self, user, history=()):
        return [self.bm.by_id[i]['text'] for i, _ in self.bm.search(user, prev_user(history), k=MAX_NOTES, min_score=self.min_score)]

# ---- the visitor simulator (S4 futures, eval dialogs): the untuned teacher plays the visitor ------------------------------------------
SIM_SYS = ('You are role-playing a visitor to Astro Pilot, a spaceflight demo running in the browser, who is chatting with its guide '
           'CAPCOM. Persona: {persona}.{style} Your hidden goal: {goal}.{intent} Write only the visitor\'s next chat message: one short '
           'message (at most 25 words), in character, reacting to what CAPCOM just said. No quotes, no name labels.')

def sim_messages(turns, persona='curious visitor', goal='find out what this demo is', style='', intent=''):
    """Messages for the simulator; turns = the CAPCOM chat so far [{'role': 'user' (the visitor) | 'assistant' (CAPCOM), 'content'}]."""
    sys = SIM_SYS.format(persona=persona or 'curious visitor', goal=(goal or 'explore the demo').rstrip('.'),
                         style=f' Writing style: {style}.' if style else '', intent=f' What you want next: {intent}.' if intent else '')
    chat = '\n'.join(f"{'Visitor' if m['role'] == 'user' else 'CAPCOM'}: {m['content']}" for m in turns[-6:])
    return [{'role': 'system', 'content': sys}, {'role': 'user', 'content': f"The chat so far:\n{chat}\n\nWrite the visitor's next message."}]

def clean_visitor(text, fallback='ok, what else can I try?'):
    """The first non-empty line without labels/quotes, at most 40 words; fallback when nothing usable is left."""
    for line in (text or '').splitlines():
        line = line.strip().strip('"\'').strip()
        for lab in ('Visitor:', 'visitor:', 'User:', 'user:', 'Me:'):
            if line.startswith(lab): line = line[len(lab):].strip().strip('"\'')
        if line and not line.lower().startswith('capcom'): return ' '.join(line.split()[:40])
    return fallback

# ---- models and generation ----------------------------------------------------------------------------------------------------------
def load_tok(path, side='left'):
    from transformers import AutoTokenizer
    t = AutoTokenizer.from_pretrained(str(path)); t.padding_side = side
    if t.pad_token is None: t.pad_token = t.eos_token
    return t

def infer_dtype(info, student=False):
    """Inference dtype: bf16 on Ampere+; on T4 fp16 for the teacher/simulator (memory) but fp32 for the small student (no fp16 overflow
    risk where the scores are measured); fp32 on MPS/CPU."""
    import torch
    if info['device'] != 'cuda': return torch.float32
    return torch.bfloat16 if info['bf16'] else torch.float32 if student else torch.float16

def load_model(path, info, dtype=None):
    """A causal LM on the probed device (SDPA attention); dtype default: fp32 (trainable master weights)."""
    import torch
    from transformers import AutoModelForCausalLM
    kw = {'dtype': dtype or torch.float32}
    try: m = AutoModelForCausalLM.from_pretrained(str(path), attn_implementation='sdpa', **kw)
    except (ValueError, ImportError): m = AutoModelForCausalLM.from_pretrained(str(path), **kw)
    return m.to(info['device'])

def eot_ids(tok):
    """End-of-turn token ids: eos plus <|im_end|> when the vocabulary has it."""
    ids = {tok.eos_token_id}
    im = tok.convert_tokens_to_ids('<|im_end|>')
    if isinstance(im, int) and im != tok.unk_token_id: ids.add(im)
    return sorted(i for i in ids if i is not None)

def chat_ids(tok, messages):
    """The prompt's token ids exactly as generation sees them (chat template + generation prompt, no extra specials)."""
    return tok(tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False), add_special_tokens=False)['input_ids']

def generate(model, tok, prompts, n=1, bs=16, max_new_tokens=120, do_sample=False, temperature=1.0, top_p=1.0,
             repetition_penalty=1.1, seed=None, max_prompt=None):
    """Batched chat generation (left padding, longest prompts first). prompts: message lists. Returns [str] (n == 1) or [[str] * n]."""
    import torch
    from transformers import GenerationConfig
    if not prompts: return []
    if seed is not None: torch.manual_seed(seed)
    ids = [chat_ids(tok, p) for p in prompts]
    if max_prompt: ids = [x[-max_prompt:] for x in ids]
    order = sorted(range(len(ids)), key=lambda i: -len(ids[i])); out = [None] * len(ids)
    gc_ = GenerationConfig(max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature if do_sample else 1.0,
                           top_p=top_p if do_sample else 1.0, top_k=0 if do_sample else 50, min_p=None, repetition_penalty=repetition_penalty,
                           num_return_sequences=n, pad_token_id=tok.pad_token_id, eos_token_id=eot_ids(tok))
    was, per = model.training, max(1, bs // n); model.eval()
    with torch.no_grad():
        for s in range(0, len(order), per):
            idx = order[s:s + per]; L = max(len(ids[i]) for i in idx)
            x = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long); m = torch.zeros_like(x)
            for r, i in enumerate(idx): x[r, L - len(ids[i]):] = torch.tensor(ids[i]); m[r, L - len(ids[i]):] = 1
            g = model.generate(input_ids=x.to(model.device), attention_mask=m.to(model.device), generation_config=gc_)
            texts = tok.batch_decode(g[:, L:], skip_special_tokens=True)
            for r, i in enumerate(idx):
                t = [s_.strip() for s_ in texts[r * n:(r + 1) * n]]; out[i] = t if n > 1 else t[0]
    if was: model.train()
    return out

# ---- the run: local work dir + Drive mirror, DONE markers, resume --------------------------------------------------------------------
def copy_tree(src, dst):
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('._*', '.DS_Store'))

def keep_model(src, dst, info):
    """Copy a model folder (or save a Hub model) to dst — a stage that keeps its input unchanged."""
    if Path(str(src)).is_dir(): copy_tree(src, dst); return
    import torch
    m, t = load_model(src, info, dtype=torch.bfloat16), load_tok(src)
    m.save_pretrained(str(dst)); t.save_pretrained(str(dst))

def latest_ckpt(d, need_mark=False):
    """The newest checkpoint-<step> folder under d (on Drive only complete copies, marked by .complete)."""
    d = Path(d)
    if not d.exists(): return None
    c = [p for p in d.glob('checkpoint-*') if p.is_dir() and p.name.split('-')[-1].isdigit() and (not need_mark or (p / '.complete').exists())]
    return max(c, key=lambda p: int(p.name.split('-')[-1])) if c else None

STUDENT_STAGES = ('S2', 'S3', 'S4', 'S5', 'S6')

class Run:
    """One training run: work/<stage>/ holds checkpoints and the stage's final `model/`; drive/<stage>/ mirrors the model, the newest
    checkpoint (ckpt/) and DONE.json, so a Colab disconnect resumes where it stopped; evals/eval_<stage>.json on both."""
    def __init__(self, work, drive=None, pkg=None, info=None, student='lfm350', teacher_size='1.2b', teacher_override=None,
                 smoke=False, seed=0, general_frac=GENERAL_FRAC):
        self.work, self.drive = Path(work), (Path(drive) if drive else None)
        self.pkg = Path(pkg) if pkg else None; self.info = info or probe(); self.tier = self.info['tier']
        self.student, self.smoke, self.seed, self.general_frac = student, bool(smoke), seed, general_frac
        m = MODELS[student]; self.student_id = m['student']
        self.teacher_id = teacher_override or m['teacher_big' if teacher_size == '2.6b' else 'teacher']
        self.sim_id = self.teacher_id  # the untuned teacher plays the visitor: the S1 LoRA made it a CAPCOM, not a visitor
        self.kb = json.loads((self.pkg / 'kb.json').read_text()) if self.pkg and (self.pkg / 'kb.json').exists() else None
        self.work.mkdir(parents=True, exist_ok=True)
        if self.drive: self.drive.mkdir(parents=True, exist_ok=True)

    def hp(self, stage): return hp(stage, self.tier, self.smoke, self.student)
    def prec(self): return {'bf16': self.info['bf16'], 'fp16': self.info['fp16']}
    def dir(self, stage):
        p = self.work / stage; p.mkdir(parents=True, exist_ok=True); return p
    def ddir(self, stage): return self.drive / stage if self.drive else None

    def done(self, stage):
        """The DONE.json info of a finished stage (Drive first, then local) or None."""
        for base in (self.drive, self.work):
            if base and (base / stage / 'DONE.json').exists(): return json.loads((base / stage / 'DONE.json').read_text())
        return None

    def finish(self, stage, **info):
        """Mirror work/<stage>/model to Drive, drop the stage's Drive checkpoints, then write DONE.json (last: a partial copy never counts)."""
        src = self.work / stage / 'model'; info = {'stage': stage, 'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'has_model': src.exists(), **info}
        if self.drive:
            d = self.ddir(stage); d.mkdir(parents=True, exist_ok=True)
            if src.exists(): shutil.rmtree(d / 'model', ignore_errors=True); copy_tree(src, d / 'model')
            for f in self.dir(stage).glob('*.json*'):  # small artefacts (pairs.jsonl, ...)
                if f.name != 'DONE.json' and not f.name.startswith('._'): shutil.copy(f, d / f.name)
            shutil.rmtree(d / 'ckpt', ignore_errors=True)
            (d / 'DONE.json').write_text(json.dumps(info, indent=1))
        self.dir(stage).joinpath('DONE.json').write_text(json.dumps(info, indent=1))
        log(f'{stage} done', json.dumps({k: v for k, v in info.items() if k not in ('stage', 'finished')})[:300])
        return info

    def model(self, stage):
        """Local folder of a finished stage's model (restored from Drive after a disconnect), or None."""
        if not (self.done(stage) or {}).get('has_model'): return None
        loc = self.work / stage / 'model'
        if not (loc / 'config.json').exists() and self.drive and (self.ddir(stage) / 'model' / 'config.json').exists():
            log(f'restoring {stage} model from Drive'); copy_tree(self.ddir(stage) / 'model', loc)
        return loc if (loc / 'config.json').exists() else None

    def resume(self, stage, out):
        """Checkpoint to resume a stage's trainer from: the newest local one, else the Drive mirror copied back."""
        ck = latest_ckpt(out)
        if ck is None and self.drive:
            dck = latest_ckpt(self.ddir(stage) / 'ckpt', need_mark=True)
            if dck: log(f'restoring {stage} checkpoint {dck.name} from Drive'); copy_tree(dck, Path(out) / dck.name); ck = Path(out) / dck.name
        return str(ck) if ck else None

    def mirror_callback(self, stage):
        """A TrainerCallback copying each saved checkpoint to drive/<stage>/ckpt (marked complete, older copies removed)."""
        from transformers import TrainerCallback
        run = self
        class Mirror(TrainerCallback):
            def on_save(self, args, state, control, **kw):
                ck = Path(args.output_dir) / f'checkpoint-{state.global_step}'
                if not run.drive or not ck.exists(): return
                dst = run.ddir(stage) / 'ckpt'; copy_tree(ck, dst / ck.name); (dst / ck.name / '.complete').write_text('ok')
                for old in dst.glob('checkpoint-*'):
                    if old.name != ck.name: shutil.rmtree(old, ignore_errors=True)
        return Mirror()

    def targs(self, cls, stage, out, h, **kw):
        """TRL/Trainer config with the run's precision, batch sizes, checkpointing and logging (kw override)."""
        smoke = self.smoke; save = h.get('save', 0)
        a = dict(output_dir=str(out), per_device_train_batch_size=h['bs'], gradient_accumulation_steps=h['ga'], learning_rate=h['lr'],
                 num_train_epochs=h.get('epochs', 1), max_steps=h.get('max_steps', -1), logging_steps=1 if smoke else 10,
                 save_strategy='steps' if save else 'no', save_steps=save or 500, save_total_limit=1, report_to='none', seed=self.seed,
                 bf16=self.prec()['bf16'], fp16=self.prec()['fp16'], gradient_checkpointing=True, dataloader_num_workers=0,
                 dataloader_pin_memory=self.info['device'] == 'cuda', use_cpu=self.tier == 'cpu', disable_tqdm=smoke,
                 lr_scheduler_type='cosine', warmup_steps=h.get('warmup', 0.03))
        a.update(kw)
        return cls(**a)

    def save_eval(self, stage, report):
        for base in (self.work, self.drive):
            if base: (base / 'evals').mkdir(parents=True, exist_ok=True); (base / 'evals' / f'eval_{stage}.json').write_text(json.dumps(report, indent=1))

    def evals(self):
        """{stage: eval report} from Drive (else local)."""
        out = {}
        for base in (self.work, self.drive):
            if base and (base / 'evals').exists():
                for p in sorted((base / 'evals').glob('eval_*.json')):
                    if not p.name.startswith('._'): out[p.stem[5:]] = json.loads(p.read_text())
        return dict(sorted(out.items(), key=lambda kv: (kv[0] != 'base', kv[0])))

    def latest(self, before=None):
        """(stage, folder) of the newest finished student stage before `before`, else ('base', the Hub id)."""
        stages = STUDENT_STAGES if before is None else STUDENT_STAGES[:STUDENT_STAGES.index(before)]
        for s in reversed(stages):
            m = self.model(s)
            if m: return s, m
        return 'base', self.student_id

    def best(self, before=None):
        """(stage, folder) of the best-scoring evaluated student stage before `before` (R5: a stage is kept only if it improves the eval
        score), falling back to latest()."""
        ev, stages = self.evals(), (STUDENT_STAGES if before is None else STUDENT_STAGES[:STUDENT_STAGES.index(before)])
        cand = [(ev[s]['score'], i, s) for i, s in enumerate(stages) if s in ev and ev[s].get('score') is not None and self.model(s)]
        if not cand: return self.latest(before)
        s = max(cand)[2]
        return s, self.model(s)

    def final(self):
        """(stage, folder) of the model to ship: S6's pick, else best()."""
        d = self.done('S6')
        if d and self.model(d['final']): return d['final'], self.model(d['final'])
        return self.best()

def cli(doc, extra=()):
    """argparse for the stages' local CLIs: returns (Run, args). extra: [(flag, kwargs)]."""
    import argparse
    os.environ.setdefault('HF_HOME', '/Volumes/LaCie/astro-pilot/chat/hf')
    ap = argparse.ArgumentParser(description=doc.split('\n')[0])
    ap.add_argument('--pkg', required=True, help='dataset package folder (or its parent)'); ap.add_argument('--work', required=True)
    ap.add_argument('--drive', help='mirror folder (Drive)'); ap.add_argument('--student', default='lfm350', choices=sorted(MODELS))
    ap.add_argument('--teacher-size', default='1.2b', choices=['1.2b', '2.6b']); ap.add_argument('--teacher', help='teacher id/path override')
    ap.add_argument('--smoke', action='store_true'); ap.add_argument('--seed', type=int, default=0)
    for flag, kw in extra: ap.add_argument(flag, **kw)
    a = ap.parse_args()
    return Run(a.work, a.drive, find_package(a.pkg), None, a.student, a.teacher_size, a.teacher, a.smoke, a.seed), a


In [ ]:
%%writefile {CODE}/chat/train/sft_stage.py
"""chat/train/sft_stage.py — S1 teacher SFT and S2 student SFT (spec §6, R5).
S1: LoRA r 64 / alpha 128 on all linear layers of the teacher, lr 1e-4, 2 epochs on A100 (1 on T4/L4), then merge_and_unload -> a full
    HF model (bf16) in S1/model — the on-policy distillation teacher of S3.
S2: full fine-tune of the student, lr 3e-5 cosine (5e-5 for SmolLM2-135M), warmup 3 %, 3 epochs (2 on T4), effective batch 64, max length
    1024, bf16 mixed precision over fp32 master weights (fp16 on T4, fp32 on MPS/CPU).
Rows are TRL prompt-completion rows (chat/data/build.py) + GENERAL_FRAC general replay, tokenized here by pc_tokens so the loss covers
exactly the assistant completion and its end-of-turn token — never the prompt, the history turns (written under other notes) or the
newline the chat template appends after <|im_end|> — and handed to trl.SFTTrainer pre-tokenized (input_ids + labels).
  python -m chat.train.sft_stage --stage s2 --pkg <package> --work <dir> [--drive <dir>] [--smoke] [--teacher <id>]"""
import torch
from chat.train.common import cli, free, load_model, load_tok, log, seed_all, sft_rows, infer_dtype

def pc_tokens(tok, prompt, completion, max_len=None):
    """{'input_ids', 'labels'} of a prompt-completion row; labels are -100 on the prompt and cover the completion text + end-of-turn
    token. The prompt ids equal generation's (template + generation prompt); None when the row exceeds max_len."""
    p = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    full = tok.apply_chat_template(prompt + completion, tokenize=False)
    if not full.startswith(p): raise ValueError('chat template is not prefix-preserving for this row')
    tail, eos = full[len(p):], tok.eos_token
    i = tail.find(eos)
    tail = tail[:i + len(eos)] if i >= 0 else tail.rstrip() + eos
    ip, ic = tok(p, add_special_tokens=False)['input_ids'], tok(tail, add_special_tokens=False)['input_ids']
    if max_len and len(ip) + len(ic) > max_len: return None
    return {'input_ids': ip + ic, 'labels': [-100] * len(ip) + ic}

def tokenize_rows(tok, rows, max_len):
    """(tokenized rows with a 'length' column, {'too_long': n, 'bad': n})."""
    out, drop = [], {'too_long': 0, 'bad': 0}
    for r in rows:
        try: t = pc_tokens(tok, r['prompt'], r['completion'], max_len)
        except Exception: drop['bad'] += 1; continue
        if t is None: drop['too_long'] += 1; continue
        t['length'] = len(t['input_ids']); out.append(t)
    return out, drop

def save_model(model, tok, dst, dtype=torch.bfloat16):
    """Save a full HF model (bf16 by default: half the Drive space; later stages reload fp32 master weights) + tokenizer."""
    model.to(dtype).save_pretrained(str(dst)); tok.save_pretrained(str(dst))

def _sft(run, stage, base, h, lora):
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    seed_all(run.seed); out = run.dir(stage); tok = load_tok(base, 'right')
    rows = sft_rows(run.pkg, 'train', run.general_frac, h['max_rows'], run.seed)
    data, drop = tokenize_rows(tok, rows, h['max_len'])
    val, _ = tokenize_rows(tok, sft_rows(run.pkg, 'val', 0, 8 if run.smoke else 256, run.seed), h['max_len'])
    if not data: raise RuntimeError(f'{stage}: no trainable rows ({drop})')
    log(f'{stage}: {len(data)} rows ({drop}), {len(val)} val rows, base {base}, hp {h}')
    frozen = lora and run.info['device'] == 'cuda'  # LoRA: frozen base in bf16/fp16, fp32 adapters; full FT: fp32 master weights
    model = load_model(base, run.info, dtype=infer_dtype(run.info) if frozen else torch.float32)
    peft = None
    if lora:
        from peft import LoraConfig
        peft = LoraConfig(r=h['lora_r'], lora_alpha=h['lora_alpha'], lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
    cfg = run.targs(SFTConfig, stage, out, h, max_length=h['max_len'], packing=False, train_sampling_strategy='group_by_length',
                    warmup_steps=h.get('warmup', 0.03), eval_strategy='no', per_device_eval_batch_size=h['bs'])
    tr = SFTTrainer(model=model, args=cfg, train_dataset=Dataset.from_list(data), eval_dataset=Dataset.from_list(val) if val else None,
                    processing_class=tok, peft_config=peft, callbacks=[run.mirror_callback(stage)])
    res = tr.train(resume_from_checkpoint=run.resume(stage, out))
    ev = tr.evaluate() if val else {}
    m = tr.model.merge_and_unload() if lora else tr.model
    save_model(m, tok, out / 'model')
    info = {'base': str(base), 'rows': len(data), 'dropped': drop, 'train_loss': round(res.training_loss, 4),
            'val_loss': round(ev['eval_loss'], 4) if 'eval_loss' in ev else None, 'steps': res.global_step}
    del tr, model, m; free()
    return info

def teacher_sft(run):
    """S1: LoRA SFT of run.teacher_id on the CAPCOM rows, merged -> S1/model. Skipped when DONE."""
    if run.done('S1'): return run.model('S1')
    info = _sft(run, 'S1', run.teacher_id, run.hp('s1'), lora=True)
    run.finish('S1', **info)
    return run.model('S1')

def student_sft(run):
    """S2: full fine-tune of run.student_id -> S2/model. Skipped when DONE."""
    if run.done('S2'): return run.model('S2')
    info = _sft(run, 'S2', run.student_id, run.hp('s2'), lora=False)
    run.finish('S2', **info)
    return run.model('S2')

if __name__ == '__main__':
    run, a = cli(__doc__, [('--stage', dict(choices=['s1', 's2'], required=True))])
    print((teacher_sft if a.stage == 's1' else student_sft)(run))


In [ ]:
%%writefile {CODE}/chat/train/distill_stage.py
"""chat/train/distill_stage.py — S3 on-policy distillation (spec §6, R5; research: OPD beats RL ~10x for small students).
trl.DistillationTrainer: the student samples its own replies to prompts/train.jsonl contexts (conversational prompt-only rows: system
notes + state, history, visitor message) and matches the S1 teacher's next-token distribution on them — beta 1.0 (reverse KL),
temperature 1.0, lr 1e-5, max completion 128 tokens, 400 steps x 32 prompts on A100 (tier-scaled). The student starts from the best
evaluated stage so far (S2); the teacher is S1/model (the untuned teacher when S1 is off). Teacher and student share the vocabulary.
  python -m chat.train.distill_stage --pkg <package> --work <dir> [--drive <dir>] [--smoke] [--teacher <id>]"""
import torch
from chat.train.common import cli, free, infer_dtype, load_model, load_tok, log, prompt_rows, seed_all
from chat.train.sft_stage import save_model

def distill(run):
    """S3 -> S3/model. Skipped when DONE."""
    if run.done('S3'): return run.model('S3')
    from datasets import Dataset
    from trl import DistillationConfig, DistillationTrainer
    h, out = run.hp('s3'), run.dir('S3'); seed_all(run.seed)
    src_stage, src = run.best('S3'); teacher = run.model('S1') or run.teacher_id
    rows = prompt_rows(run.pkg, 'train', h['n_prompts'], run.seed)
    if not rows: raise RuntimeError('S3: prompts/train.jsonl is empty')
    log(f'S3: {len(rows)} prompts, student {src_stage} ({src}), teacher {teacher}, hp {h}')
    tok = load_tok(src, 'left')
    model = load_model(src, run.info, dtype=torch.float32)
    tmodel = load_model(teacher, run.info, dtype=infer_dtype(run.info)).eval()
    cfg = run.targs(DistillationConfig, 'S3', out, h, beta=h['beta'], temperature=h['temperature'], max_completion_length=h['max_completion'],
                    num_train_epochs=100, max_steps=h['max_steps'], warmup_steps=0.05, disable_dropout=True, use_vllm=False)
    tr = DistillationTrainer(model=model, teacher_model=tmodel, args=cfg, train_dataset=Dataset.from_list([{'prompt': r['prompt']} for r in rows]),
                             processing_class=tok, callbacks=[run.mirror_callback('S3')])
    res = tr.train(resume_from_checkpoint=run.resume('S3', out))
    save_model(tr.model, tok, out / 'model')
    info = {'student_from': src_stage, 'teacher': str(teacher), 'prompts': len(rows), 'steps': res.global_step,
            'train_loss': round(res.training_loss, 4)}
    del tr, model, tmodel; free()
    run.finish('S3', **info)
    return run.model('S3')

if __name__ == '__main__':
    run, _ = cli(__doc__)
    print(distill(run))


In [ ]:
%%writefile {CODE}/chat/train/dpo_stage.py
"""chat/train/dpo_stage.py — S4 multi-turn-aware DPO (spec §6, R5/R6; CollabLLM-style 2-turn simulated futures).
Per round: N prompt contexts (prompts/train.jsonl), K = 4 policy samples each (temperature 0.8). For every sample the untuned teacher
plays the visitor (meta.persona + meta.goal + the chat so far incl. the sample -> one short message), the page's BM25 retrieves notes
for that message (kb.json min_score), chat.prompt.messages builds the next prompt and the policy answers greedily. Reward =
rewards.score(sample, row).total + FUTURE_W * rewards.score(next_reply, next_row).total (next_row: the simulated message and its notes,
must [], abstain False, lead True). chosen = best, rejected = worst sample, pairs kept when the margin >= 0.15. Then trl.DPOTrainer with
loss_type ['apo_zero'], beta 0.3, lr 1e-6, 2 epochs. Rounds chain (round r trains on round r-1's samples' winner); the last round's
output is S4/model, earlier rounds are S4_<r>/model. N and K scale with the tier; T4 runs one round. Generation is batched; a round's
pairs are cached in pairs.json (copied to Drive at once), so a disconnect during the DPO step does not repeat the rollouts.
  python -m chat.train.dpo_stage --pkg <package> --work <dir> [--drive <dir>] [--smoke] [--teacher <id>]"""
import json
import torch
from chat.prompt import messages
from chat.rewards import score
from chat.train.common import (DEFAULT_STATE, Notes, clean_visitor, cli, free, generate, infer_dtype, keep_model, load_model, load_tok,
                               log, prompt_rows, seed_all, sim_messages)
from chat.train.sft_stage import save_model

FUTURE_W = 0.5

def history(row, reply=None):
    """The chat so far without the system message (+ the candidate reply)."""
    h = [{'role': m['role'], 'content': m['content']} for m in row['prompt'] if m['role'] != 'system']
    return h + ([{'role': 'assistant', 'content': reply}] if reply is not None else [])

def rollouts(policy, ptok, sim, stok, rows, notes, h, seed=0):
    """[[{'text', 'score', 'now', 'future', 'visitor', 'next'} x K] per row]: K samples, their simulated visitor turn and next reply."""
    k = h['k']
    cands = generate(policy, ptok, [r['prompt'] for r in rows], n=k, bs=h['gen_bs'], max_new_tokens=h['max_new'], do_sample=True,
                     temperature=h['temperature'], repetition_penalty=1.0, seed=seed)
    cands = [c if isinstance(c, list) else [c] for c in cands]
    flat = [(i, j) for i in range(len(rows)) for j in range(k)]
    meta = [rows[i].get('meta') or {} for i, _ in flat]
    sp = [sim_messages(history(rows[i], cands[i][j]), m.get('persona'), m.get('goal')) for (i, j), m in zip(flat, meta)]
    visitors = [clean_visitor(v) for v in generate(sim, stok, sp, bs=h['gen_bs'], max_new_tokens=h['sim_tokens'], repetition_penalty=1.1)]
    nrows = []
    for (i, j), m, v in zip(flat, meta, visitors):
        hist = history(rows[i], cands[i][j]); ns = notes(v, hist)
        nrows.append({'prompt': messages(hist, v, ns, m.get('state') or DEFAULT_STATE), 'notes': ns, 'user': v, 'must': [], 'abstain': False, 'lead': True})
    nexts = generate(policy, ptok, [r['prompt'] for r in nrows], bs=h['gen_bs'], max_new_tokens=h['max_new'], repetition_penalty=1.1)
    groups = [[] for _ in rows]
    for f, (i, j) in enumerate(flat):
        now, fut = score(cands[i][j], rows[i])['total'], score(nexts[f], nrows[f])['total']
        groups[i].append({'text': cands[i][j], 'score': now + FUTURE_W * fut, 'now': now, 'future': fut, 'visitor': visitors[f], 'next': nexts[f]})
    return groups

def select_pairs(groups, margin=0.15):
    """[(i, chosen, rejected, margin)]: the best vs the worst distinct sample of each group when best - worst >= margin (> 0)."""
    out = []
    for i, g in enumerate(groups):
        if len(g) < 2: continue
        b, w = max(g, key=lambda c: c['score']), min(g, key=lambda c: c['score'])
        d = b['score'] - w['score']
        if d > 0 and d >= margin and b['text'].strip() and b['text'] != w['text']: out.append((i, b['text'], w['text'], round(d, 4)))
    return out

def make_pairs(run, stage, src, rows, h, seed):
    """(pairs [{'prompt', 'user', 'chosen', 'rejected', 'margin'}], info) from rollouts of the policy at src; cached as <stage>/pairs.json
    (copied to Drive at once) so a disconnect during the DPO step does not repeat the rollouts."""
    for p in (run.dir(stage) / 'pairs.json', run.ddir(stage) / 'pairs.json' if run.drive else None):
        if p and p.exists():
            d = json.loads(p.read_text())
            if d['info']['policy'] == str(src): log(f"{stage}: reusing {len(d['pairs'])} cached pairs"); return d['pairs'], d['info']
    ptok = load_tok(src, 'left'); policy = load_model(src, run.info, dtype=infer_dtype(run.info, student=True)).eval()
    stok = load_tok(run.sim_id, 'left'); sim = load_model(run.sim_id, run.info, dtype=infer_dtype(run.info)).eval()
    groups = rollouts(policy, ptok, sim, stok, rows, Notes(run.kb), h, seed)
    del policy, sim; free()
    sel, fallback = select_pairs(groups, h['margin']), False
    if not sel and run.smoke:  # smoke only: guarantee a pair so the DPO step is exercised
        fallback = True
        sel = [(i, rows[i]['reference'], min(g, key=lambda c: c['score'])['text'], 0.0) for i, g in enumerate(groups) if rows[i].get('reference')][:2]
    pairs = [{'prompt': rows[i]['prompt'], 'user': rows[i]['user'], 'chosen': c, 'rejected': r, 'margin': d} for i, c, r, d in sel]
    mean = lambda xs: round(sum(xs) / len(xs), 4) if xs else None
    info = {'policy': str(src), 'contexts': len(rows), 'samples': sum(len(g) for g in groups), 'pairs': len(pairs), 'smoke_fallback': fallback,
            'mean_score': mean([c['score'] for g in groups for c in g]), 'mean_future': mean([c['future'] for g in groups for c in g]),
            'mean_margin': mean([p['margin'] for p in pairs])}
    blob = json.dumps({'info': info, 'pairs': pairs, 'examples': [g[:2] for g in groups[:20]]}, ensure_ascii=False)
    (run.dir(stage) / 'pairs.json').write_text(blob)
    if run.drive: run.ddir(stage).mkdir(parents=True, exist_ok=True); (run.ddir(stage) / 'pairs.json').write_text(blob)
    return pairs, info

def dpo_round(run, stage, src, rows, h, seed):
    """One round -> <stage>/model: pairs from rollouts of the policy at src, then DPO (apo_zero) from src."""
    from datasets import Dataset
    from trl import DPOConfig, DPOTrainer
    out = run.dir(stage); seed_all(seed)
    pairs, info = make_pairs(run, stage, src, rows, h, seed)
    log(f'{stage}: {json.dumps(info)}')
    if not pairs: log(f'{stage}: no pairs above the margin, the policy is kept'); return info | {'trained': False}
    ds = Dataset.from_list([{'prompt': p['prompt'], 'chosen': [{'role': 'assistant', 'content': p['chosen']}],
                             'rejected': [{'role': 'assistant', 'content': p['rejected']}]} for p in pairs])
    tok = load_tok(src, 'right')
    model = load_model(src, run.info, dtype=torch.float32); ref = load_model(src, run.info, dtype=infer_dtype(run.info, student=True)).eval()
    cfg = run.targs(DPOConfig, stage, out, h, beta=h['beta'], loss_type=['apo_zero'], max_length=h['max_len'], warmup_steps=0.1,
                    precompute_ref_log_probs=False)
    tr = DPOTrainer(model=model, ref_model=ref, args=cfg, train_dataset=ds, processing_class=tok, callbacks=[run.mirror_callback(stage)])
    res = tr.train(resume_from_checkpoint=run.resume(stage, out))
    save_model(tr.model, tok, out / 'model')
    info |= {'trained': True, 'steps': res.global_step, 'train_loss': round(res.training_loss, 4)}
    del tr, model, ref; free()
    return info

def dpo(run):
    """S4: h['rounds'] rounds (S4_1 .. S4); starts from the best evaluated stage so far (S3). Skipped when DONE."""
    if run.done('S4'): return run.model('S4')
    h = run.hp('s4'); R = h['rounds']; src_stage, src = run.best('S4')
    log(f'S4: {R} round(s) from {src_stage} ({src}), simulator {run.sim_id}, hp {h}')
    for r in range(1, R + 1):
        stage = 'S4' if r == R else f'S4_{r}'
        if run.done(stage):
            src = run.model(stage) or src; continue
        rows = prompt_rows(run.pkg, 'train', h['n_ctx'], seed=run.seed + 100 * r)
        info = dpo_round(run, stage, src, rows, h, seed=run.seed + r)
        if not info['trained']: keep_model(src, run.dir(stage) / 'model', run.info)  # no pairs: the policy carries forward
        run.finish(stage, round=r, **info)
        src = run.model(stage)
    return run.model('S4')

if __name__ == '__main__':
    run, _ = cli(__doc__)
    print(dpo(run))


In [ ]:
%%writefile {CODE}/chat/train/grpo_stage.py
"""chat/train/grpo_stage.py — S5 optional rubric GRPO (spec §6, R6: verifiable rewards only; off by default on T4).
trl.GRPOTrainer, loss_type 'dapo', scale_rewards 'batch', num_generations 4 (2 on T4), beta 0.02, epsilon_high 0.28, lr 1e-6, 200 steps,
max completion 128, mask_truncated_completions. One reward function per chat.rewards.score component — recall, grounded, lead, first,
length, abstain, clean (min of repetition and leak) — weighted as in rewards.W and aggregated with multi_objective_aggregation
'normalize_then_sum' (GDPO: each reward normalised within its group before the weighted sum; TRL 1.14.1 supports it). A component
that does not apply to a row (recall on an abstain row) returns None -> NaN, which TRL's nan-aware aggregation ignores. Rows come from
prompts/train.jsonl with the scorer fields (notes, user, must, abstain, lead) as dataset columns, passed to the reward functions.
  python -m chat.train.grpo_stage --pkg <package> --work <dir> [--drive <dir>] [--smoke]"""
import torch
from chat.rewards import W, score
from chat.train.common import cli, free, load_model, load_tok, log, prompt_rows, seed_all
from chat.train.sft_stage import save_model

COMPONENTS = ('recall', 'grounded', 'lead', 'first', 'length', 'abstain', 'clean')

def text_of(c):
    """A completion as text (conversational: the last message's content)."""
    return c[-1]['content'] if isinstance(c, list) else c

def component(s, name):
    return min(s['repetition'], s['leak']) if name == 'clean' else s[name]

def reward_funcs():
    """([reward_<component>(prompts, completions, notes, user, must, abstain, lead, **kw) -> [float | None]], weights)."""
    fns = []
    for name in COMPONENTS:
        def f(prompts, completions, notes, user, must, abstain, lead, _name=name, **kw):
            return [component(score(text_of(c), {'notes': n, 'user': u, 'must': m or [], 'abstain': a, 'lead': l}), _name)
                    for c, n, u, m, a, l in zip(completions, notes, user, must, abstain, lead)]
        f.__name__ = f'reward_{name}'; fns.append(f)
    return fns, [W['repetition'] + W['leak'] if n == 'clean' else W[n] for n in COMPONENTS]

def grpo(run):
    """S5 -> S5/model, from the best evaluated stage so far (S4). Skipped when DONE."""
    if run.done('S5'): return run.model('S5')
    from datasets import Dataset
    from trl import GRPOConfig, GRPOTrainer
    h, out = run.hp('s5'), run.dir('S5'); seed_all(run.seed)
    src_stage, src = run.best('S5')
    rows = prompt_rows(run.pkg, 'train', h['n_prompts'], seed=run.seed + 5)
    ds = Dataset.from_list([{'prompt': r['prompt'], 'notes': r['notes'], 'user': r['user'], 'must': r.get('must') or [],
                             'abstain': bool(r.get('abstain')), 'lead': bool(r.get('lead', True))} for r in rows])
    log(f'S5: {len(rows)} prompts from {src_stage} ({src}), hp {h}')
    fns, weights = reward_funcs()
    tok = load_tok(src, 'left'); model = load_model(src, run.info, dtype=torch.float32)
    cfg = run.targs(GRPOConfig, 'S5', out, h, loss_type='dapo', scale_rewards='batch', num_generations=h['num_generations'], beta=h['beta'],
                    epsilon_high=h['epsilon_high'], max_completion_length=h['max_completion'], mask_truncated_completions=True,
                    multi_objective_aggregation='normalize_then_sum', reward_weights=weights, temperature=1.0, num_train_epochs=100,
                    warmup_steps=0.05, lr_scheduler_type='constant_with_warmup', use_vllm=False,
                    model_init_kwargs={'dtype': 'bfloat16' if run.info['bf16'] else 'float32'})  # the reference model
    tr = GRPOTrainer(model=model, reward_funcs=fns, args=cfg, train_dataset=ds, processing_class=tok, callbacks=[run.mirror_callback('S5')])
    res = tr.train(resume_from_checkpoint=run.resume('S5', out))
    save_model(tr.model, tok, out / 'model')
    info = {'policy_from': src_stage, 'prompts': len(rows), 'steps': res.global_step, 'train_loss': round(res.training_loss, 4)}
    del tr, model; free()
    run.finish('S5', **info)
    return run.model('S5')

if __name__ == '__main__':
    run, _ = cli(__doc__)
    print(grpo(run))


In [ ]:
%%writefile {CODE}/chat/train/merge_stage.py
"""chat/train/merge_stage.py — S6 (spec §6, R5; SmolLM3's final recipe): pick the best evaluated student stage (S2..S5) by the eval score,
then try a linear interpolation 0.9 x best + 0.1 x S3 (the on-policy-distilled checkpoint) of the two state dicts; the merge is kept
only if its eval score beats the best stage. DONE.json records 'final' — the stage S7 exports.
  python -m chat.train.merge_stage --a <dir> --b <dir> --out <dir> [--w 0.9]     (a bare interpolation)"""
import torch
from chat.train.common import free, load_model, load_tok, log
from chat.train.sft_stage import save_model

CPU = {'device': 'cpu', 'bf16': False, 'fp16': False}

def interpolate(a, b, out, w=0.9):
    """out = w * a + (1 - w) * b over every floating parameter and buffer (same architecture; tied weights counted once), saved bf16 with
    a's config and tokenizer."""
    ma, mb = load_model(a, CPU, dtype=torch.float32), load_model(b, CPU, dtype=torch.float32)
    pb = {**dict(mb.named_parameters()), **dict(mb.named_buffers())}
    with torch.no_grad():
        for n, p in [*ma.named_parameters(), *ma.named_buffers()]:
            if p.is_floating_point() and n in pb: p.mul_(w).add_(pb[n].to(p.dtype), alpha=1 - w)
    save_model(ma, load_tok(a), out)
    del ma, mb; free()
    return out

def select(run, w=0.9):
    """S6 -> DONE {'final': stage, ...}; S6/model exists only when the merge won. Skipped when DONE."""
    d = run.done('S6')
    if d: return d
    from chat.eval import run_eval, table
    ev = run.evals(); best_stage, best_dir = run.best('S6'); s3 = run.model('S3')
    best_score = (ev.get(best_stage) or {}).get('score')
    info = {'best': best_stage, 'best_score': best_score, 'final': best_stage, 'merge_score': None}
    if s3 and best_stage not in ('S3', 'base'):
        out = run.dir('S6') / 'model'
        log(f'S6: merging {w} x {best_stage} + {1 - w:.1f} x S3')
        interpolate(best_dir, s3, out, w)
        rep = run_eval(out, run.pkg, run.hp('eval'), run.info, run.sim_id, run.kb); rep['stage'] = 'S6'; run.save_eval('S6', rep)
        print(table({'S6': rep}), flush=True)
        info['merge_score'] = rep['score']
        if rep['score'] is not None and (best_score is None or rep['score'] > best_score): info['final'] = 'S6'
        else:
            import shutil
            shutil.rmtree(out, ignore_errors=True)
    log(f"S6: final = {info['final']} (best {best_stage} {best_score}, merge {info['merge_score']})")
    return run.finish('S6', **info)

if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser(description='linear interpolation of two checkpoints')
    ap.add_argument('--a', required=True); ap.add_argument('--b', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--w', type=float, default=0.9)
    x = ap.parse_args(); print(interpolate(x.a, x.b, x.out, x.w))


In [ ]:
%%writefile {CODE}/chat/train/web_stage.py
"""chat/train/web_stage.py — S7 (spec §6, R7): export S6's final model with chat.export_web (ONNX Runtime GenAI builder int4 block 32:
q4f16 for WebGPU with shader-f16 + q4 for WebGPU without it — the 350M is impractical on WASM (no GatherBlockQuantized kernel in
ORT-web WASM), so the page falls back to retrieval-only cards there; ctx 4096; transformers.js IO; acc = the int4 accuracy level,
default 4 until the browser A/B decides), the greedy parity check (PyTorch fp32 vs the q4 graph on CPU),
kb.json + the eval reports + capcom.json (what was shipped) beside the model, zipped as capcom-web-<run>.zip on Drive. push() uploads
the web folder to a private Hugging Face repo (the notebook's optional last cell).
  python -m chat.train.web_stage --pkg <package> --work <dir> [--drive <dir>] [--smoke]"""
import json, shutil, zipfile
from pathlib import Path
from chat.train.common import cli, log

def web(run, dtypes=None, ctx=4096, acc=None):
    """S7 -> S7/web + capcom-web-<run>.zip (local and Drive). Skipped when DONE."""
    d = run.done('S7')
    if d: return d
    from chat.export_web import check, export
    stage, src = run.final(); dtypes = tuple(dtypes or (('q4',) if run.smoke else ('q4f16', 'q4')))
    out = run.dir('S7') / 'web'; shutil.rmtree(out, ignore_errors=True)
    log(f'S7: exporting {stage} ({src}) as {dtypes}, ctx {ctx}, int4 accuracy level {acc or 4}')
    sizes = export(src, out, dtypes, ctx, acc)
    parity = check(src, out) if 'q4' in dtypes else None
    log(f'S7: sizes {sizes} MB, parity {parity}')
    if run.kb: (out / 'kb.json').write_text(json.dumps(run.kb, ensure_ascii=False))
    ev = run.evals(); (out / 'evals').mkdir(exist_ok=True)
    for s, rep in ev.items(): (out / 'evals' / f'eval_{s}.json').write_text(json.dumps(rep, indent=1))
    card = {'student': run.student_id, 'stage': stage, 'eval_score': (ev.get(stage) or {}).get('score'), 'teacher': run.teacher_id,
            'dtypes': list(dtypes), 'sizes_mb': sizes, 'parity': parity, 'ctx': ctx, 'int4_accuracy_level': acc or 4,
            'runtime': 'WebGPU q4f16 (shader-f16) -> WebGPU q4 -> retrieval-only cards', 'prompt_contract': 'v1 (chat/prompt.py)',
            'kb': {k: run.kb.get(k) for k in ('name', 'version', 'retrieval')} if run.kb else None,
            'decoding': {'max_new_tokens': 120, 'repetition_penalty': 1.1, 'do_sample': False}, 'smoke': run.smoke}
    (out / 'capcom.json').write_text(json.dumps(card, indent=1))
    z = run.dir('S7') / f'capcom-web-{run.work.name}.zip'
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(out.rglob('*')):
            if p.is_file() and not p.name.startswith('._'): zf.write(p, f'capcom-web-{run.work.name}/{p.relative_to(out)}')
    if run.drive: shutil.copy(z, run.drive / z.name)
    return run.finish('S7', final=stage, sizes_mb=sizes, parity=parity, acc=acc or 4, zip=str(run.drive / z.name if run.drive else z),
                      zip_mb=round(z.stat().st_size / 2 ** 20, 1))

def push(run, repo_id, token, private=True):
    """Upload S7/web to the Hugging Face Hub (a private model repo by default); returns the repo URL."""
    from huggingface_hub import HfApi
    api = HfApi(token=token); api.create_repo(repo_id, private=private, exist_ok=True)
    api.upload_folder(folder_path=str(run.work / 'S7' / 'web'), repo_id=repo_id, commit_message=f'CAPCOM {run.work.name}',
                      ignore_patterns=['._*', '.DS_Store'])
    return f'https://huggingface.co/{repo_id}'

if __name__ == '__main__':
    run, _ = cli(__doc__)
    print(json.dumps(web(run), indent=1))


In [ ]:
%%writefile {CODE}/chat/eval.py
"""chat/eval.py — CAPCOM's held-out evaluation (spec §8, R6). evaluate(): greedy replies (max 120 new tokens, repetition penalty 1.1,
left-padded batches) to prompt rows -> chat.rewards.score -> metrics: score (mean total x 100), recall, grounded rate (grounded == 1),
lead_ok rate (exactly one lead move), nagging rate (>= 2 questions), abstain accuracy (rows that expect an abstention), false-answer
rate (an abstention expected but not given — the ship gate on eval/unanswerable), mean words. Sets: eval/single, eval/unanswerable, a
fixed seeded slice of prompts/val, and simulate_dialogs(): the eval/dialog_scripts.jsonl visitors played by the teacher (persona +
style + hidden goal + the script's next intent) for `turns` CAPCOM turns, retrieval as on the page. The overall score is the mean of the
per-set scores; S6 keeps the best stage by it. Empty or missing sets are skipped.
  python -m chat.eval --model <dir|id> --pkg <package> [--sim <dir|id>] [--out eval.json] [--smoke]"""
import json, random
from pathlib import Path
from chat.prompt import messages
from chat.rewards import questions, score
from chat.train.common import (DEFAULT_STATE, Notes, clean_visitor, find_package, free, generate, hp, infer_dtype, load_model, load_tok,
                               log, probe, prompt_rows, read_jsonl, sim_messages)

SETS = ('single', 'unanswerable', 'val', 'dialogs')

def _mean(xs):
    return round(sum(xs) / len(xs), 4) if xs else None

def metrics(results):
    """results: [{'answer', 'row', 's' (rewards.score)}] -> the metric dict."""
    if not results: return {'n': 0, 'score': None}
    S, A, R = [r['s'] for r in results], [r['answer'] for r in results], [r['row'] for r in results]
    abst = [s['abstain'] for s, row in zip(S, R) if row.get('abstain')]
    return {'n': len(S), 'score': round(100 * _mean([s['total'] for s in S]), 2), 'recall': _mean([s['recall'] for s in S if s['recall'] is not None]),
            'grounded': _mean([float(s['grounded'] == 1) for s in S]), 'lead_ok': _mean([float(s['lead'] == 1) for s in S]),
            'nagging': _mean([float(questions(a) >= 2) for a in A]), 'abstain_acc': _mean(abst),
            'false_answer': _mean([float(x == 0) for x in abst]), 'words': _mean([len(a.split()) for a in A])}

def evaluate(model, tok, rows, bs=16, max_new_tokens=120):
    """Greedy replies to prompt rows, scored: (metrics, results)."""
    answers = generate(model, tok, [r['prompt'] for r in rows], bs=bs, max_new_tokens=max_new_tokens, repetition_penalty=1.1)
    res = [{'answer': a, 'row': r, 's': score(a, r)} for a, r in zip(answers, rows)]
    return metrics(res), res

def opening(s):
    """The script's first visitor message (tolerant: opening | turns[0] user | user_turns[0])."""
    first = next((t['content'] for t in s.get('turns', []) if isinstance(t, dict) and t.get('role') == 'user'), None)
    return s.get('opening') or first or (s.get('user_turns') or [None])[0] or 'hi, what is this?'

def simulate_dialogs(policy, ptok, sim, stok, scripts, notes, turns=4, bs=16, max_new_tokens=120, sim_tokens=48):
    """Play each script for `turns` CAPCOM turns: CAPCOM answers greedily from retrieved notes, the simulator writes the next visitor
    message (intents[t] steers turn t+1). Returns (metrics over all turns + per-turn scores, transcripts)."""
    st = [{'s': s, 'hist': [], 'user': opening(s), 'log': []} for s in scripts]
    for t in range(turns):
        rows = []
        for d in st:
            ns = notes(d['user'], d['hist'])
            rows.append({'prompt': messages(d['hist'], d['user'], ns, d['s'].get('state') or DEFAULT_STATE), 'notes': ns, 'user': d['user'],
                         'must': [], 'abstain': False, 'lead': True})
        answers = generate(policy, ptok, [r['prompt'] for r in rows], bs=bs, max_new_tokens=max_new_tokens, repetition_penalty=1.1)
        for d, r, a in zip(st, rows, answers):
            d['log'].append({'turn': t, 'answer': a, 'row': r, 's': score(a, r)})
            d['hist'] += [{'role': 'user', 'content': d['user']}, {'role': 'assistant', 'content': a}]
        if t == turns - 1: break
        intents = [(d['s'].get('intents') or [])[t] if t < len(d['s'].get('intents') or []) else '' for d in st]
        sp = [sim_messages(d['hist'], d['s'].get('persona'), d['s'].get('goal'), d['s'].get('style', ''), it) for d, it in zip(st, intents)]
        vis = generate(sim, stok, sp, bs=bs, max_new_tokens=sim_tokens, repetition_penalty=1.1)
        for d, v, it in zip(st, vis, intents): d['user'] = clean_visitor(v, it or 'ok, what else can I try?')
    flat = [x for d in st for x in d['log']]
    m = metrics(flat)
    m['per_turn'] = [round(100 * _mean([x['s']['total'] for x in flat if x['turn'] == t] or [0]), 2) for t in range(turns)]
    tr = [{'id': d['s'].get('id'), 'turns': [{'visitor': x['row']['user'], 'capcom': x['answer'], 'score': round(x['s']['total'], 3)}
                                           for x in d['log']]} for d in st]
    return m, tr

def eval_sets(pkg, h):
    """{'single', 'unanswerable', 'val': prompt rows, 'dialogs': scripts} cut per h (n_* None = all; val = a fixed seeded slice)."""
    pkg = Path(pkg); cut = lambda rows, n: rows[:n] if n else rows
    val = prompt_rows(pkg, 'val', h['n_val'], seed=0)
    return {'single': cut(read_jsonl(pkg / 'eval/single.jsonl'), h['n_single']),
            'unanswerable': cut(read_jsonl(pkg / 'eval/unanswerable.jsonl'), h['n_unans']), 'val': val,
            'dialogs': cut(read_jsonl(pkg / 'eval/dialog_scripts.jsonl'), h['n_scripts'])}

def run_eval(model_path, pkg, h, info=None, sim_path=None, kb=None, samples=8):
    """The full report of one model: {'model', 'score' (mean of set scores), 'sets': {name: metrics}, 'samples', 'dialogs'}."""
    info = info or probe(); pkg = Path(pkg); sets = eval_sets(pkg, h)
    kb = kb or json.loads((pkg / 'kb.json').read_text())
    tok = load_tok(model_path, 'left'); model = load_model(model_path, info, dtype=infer_dtype(info, student=True)).eval()
    rep = {'model': str(model_path), 'sets': {}, 'samples': {}}
    for name in ('single', 'unanswerable', 'val'):
        if not sets[name]: continue
        m, res = evaluate(model, tok, sets[name], h['bs'], h['max_new'])
        rep['sets'][name] = m
        rep['samples'][name] = [{'user': r['row']['user'], 'answer': r['answer'], 'reference': r['row'].get('reference'),
                                 'total': round(r['s']['total'], 3)} for r in res[:samples]]
    if sets['dialogs'] and sim_path:
        stok = load_tok(sim_path, 'left'); sim = load_model(sim_path, info, dtype=infer_dtype(info)).eval()
        m, tr = simulate_dialogs(model, tok, sim, stok, sets['dialogs'], Notes(kb), h['turns'], h['bs'], h['max_new'])
        rep['sets']['dialogs'], rep['dialogs'] = m, tr[:samples]
        del sim
    del model; free()
    scores = [m['score'] for m in rep['sets'].values() if m.get('score') is not None]
    rep['score'] = round(sum(scores) / len(scores), 2) if scores else None
    return rep

COLS = ('score', 'single', 'unanswerable', 'val', 'dialogs', 'recall', 'grounded', 'lead_ok', 'nagging', 'false_answer', 'words')

def table(evals):
    """A compact text table of {stage: report}: set columns are set scores, false_answer is eval/unanswerable's (the ship gate), the
    rest are pooled over the single-turn sets."""
    def pooled(rep, k):
        if k == 'false_answer' and (rep['sets'].get('unanswerable') or {}).get(k) is not None: return rep['sets']['unanswerable'][k]
        ms = [m for n, m in rep['sets'].items() if n != 'dialogs' and m.get(k) is not None]
        return sum(m[k] * m['n'] for m in ms) / sum(m['n'] for m in ms) if ms else None
    fmt = lambda v: '-' if v is None else (f'{v:.1f}' if abs(v) >= 2 else f'{v:.3f}')
    lines = ['stage  ' + ' '.join(f'{c[:9]:>9}' for c in COLS)]
    for st, rep in evals.items():
        vals = [rep.get('score')] + [rep['sets'].get(n, {}).get('score') for n in SETS] + [pooled(rep, k) for k in COLS[5:]]
        lines.append(f'{st:<6} ' + ' '.join(f'{fmt(v):>9}' for v in vals))
    return '\n'.join(lines)

def eval_stage(run, stage):
    """Evaluate a finished stage's model once (cached in evals/eval_<stage>.json; 'base' = the untuned student, a reference row); None
    when the stage has no model."""
    ev = run.evals()
    if stage in ev: return ev[stage]
    path = run.student_id if stage == 'base' else run.model(stage)
    if path is None: log(f'eval {stage}: no model, skipped'); return None
    rep = run_eval(path, run.pkg, run.hp('eval'), run.info, run.sim_id, run.kb)
    rep['stage'] = stage; run.save_eval(stage, rep); print(table({stage: rep}), flush=True)
    return rep

if __name__ == '__main__':
    import argparse, os
    os.environ.setdefault('HF_HOME', '/Volumes/LaCie/astro-pilot/chat/hf')
    ap = argparse.ArgumentParser(description='CAPCOM eval'); ap.add_argument('--model', required=True); ap.add_argument('--pkg', required=True)
    ap.add_argument('--sim'); ap.add_argument('--out'); ap.add_argument('--smoke', action='store_true')
    a = ap.parse_args(); info = probe()
    rep = run_eval(a.model, find_package(a.pkg), hp('eval', info['tier'], a.smoke), info, a.sim)
    if a.out: Path(a.out).write_text(json.dumps(rep, indent=1))
    print(table({'model': rep}))


In [ ]:
# ---- import the library just written (re-running the library cells picks up changes) ----
import sys
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
for m in [m for m in sys.modules if m == 'chat' or m.startswith('chat.')]: del sys.modules[m]
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
from chat.train import common, sft_stage, distill_stage, dpo_stage, grpo_stage, merge_stage, web_stage
from chat import eval as capcom_eval
print('library ok:', common.__file__)

## GPU

In [ ]:
# ---- GPU probe -> tier (t4 | l4 | a100 | h100 | big | mps | cpu), precision, stage defaults ----
INFO = common.probe(); print(INFO)
if not LOCAL: assert INFO['device'] == 'cuda', 'Runtime > Change runtime type > pick a GPU'
if STAGES['S5'] == 'auto': STAGES['S5'] = common.default_stages(INFO['tier'])['S5']
if TEACHER_SIZE == '2.6b' and (INFO['mem_gb'] or 0) < 39: print('the 2.6b teacher needs >= 40 GB: using 1.2b'); TEACHER_SIZE = '1.2b'
print('stages:', STAGES)
for s in ('s1', 's2', 's3', 's4', 's5', 'eval'): print(s, common.hp(s, INFO['tier'], bool(SMOKE), STUDENT))

## Data

In [ ]:
# ---- Google Drive + data: copy the zip to local disk, unzip, verify every file's sha256 against MANIFEST.json ----
import shutil, zipfile
if LOCAL: DRIVE = Path(os.environ.get('CAPCOM_DRIVE') or ROOT / 'drive')
else:
    from google.colab import drive
    drive.mount('/content/drive'); DRIVE = Path(DRIVE_DIR)
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # optional: faster, un-throttled Hub downloads
    except Exception: pass
zsrc = DRIVE / DATA_ZIP
assert zsrc.exists(), f'upload {DATA_ZIP} to {DRIVE} first (see the top of the notebook)'
zloc, data = ROOT / DATA_ZIP, ROOT / 'data' / Path(DATA_ZIP).stem
if not zloc.exists() or zloc.stat().st_size != zsrc.stat().st_size: shutil.copy(zsrc, zloc)
if not (data / '.unzipped').exists():
    shutil.rmtree(data, ignore_errors=True); zipfile.ZipFile(zloc).extractall(data); (data / '.unzipped').write_text('ok')
PKG = common.find_package(data); bad = common.verify_manifest(PKG)
assert not bad, f'corrupt package: {bad}'
man = json.loads((PKG / 'MANIFEST.json').read_text())
print(PKG, man['version'], {k: v['rows'] for k, v in man['files'].items() if v['rows'] is not None})
RUN = common.Run(ROOT / 'runs' / RUN_NAME, DRIVE / 'runs' / RUN_NAME, PKG, INFO, STUDENT, TEACHER_SIZE, TEACHER_OVERRIDE, SMOKE)
(RUN.drive / 'requirements.lock').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, cwd=Path.home()).stdout)
print('student', RUN.student_id, '| teacher', RUN.teacher_id, '| run', RUN.work, '->', RUN.drive)

## Baseline
The untuned student with the same prompt contract: the reference row of the results table.

In [ ]:
capcom_eval.eval_stage(RUN, 'base')

## S1 — teacher SFT
LoRA r 64 / α 128 on every linear layer of the teacher, lr 1e-4, then merged into a full model: the distillation teacher of S3. The loss covers only the CAPCOM reply (+ end-of-turn token).

In [ ]:
if STAGES['S1']: sft_stage.teacher_sft(RUN)

## S2 — student SFT
Full fine-tune of the student: lr 3e-5 cosine, warmup 3 %, 3 epochs, effective batch 64, max length 1024; 25 % general-chat replay.

In [ ]:
if STAGES['S2']: sft_stage.student_sft(RUN)

### Eval S2
Greedy replies to the held-out sets, scored by `chat/rewards.py`; multi-turn scripts with the teacher as the visitor.

In [ ]:
capcom_eval.eval_stage(RUN, 'S2')

## S3 — on-policy distillation
`trl.DistillationTrainer`: the student samples, the S1 teacher scores every token (reverse KL, β 1.0), lr 1e-5, 128-token completions.

In [ ]:
if STAGES['S3']: distill_stage.distill(RUN)

### Eval S3

In [ ]:
capcom_eval.eval_stage(RUN, 'S3')

## S4 — multi-turn-aware DPO
For each context: 4 samples; the teacher plays the visitor's next message, the student answers it; reward = score(reply) + 0.5 × score(next reply). Best vs worst pairs (margin ≥ 0.15) → `DPOTrainer` `apo_zero`, β 0.3, lr 1e-6. Two rounds (one on T4).

In [ ]:
if STAGES['S4']: dpo_stage.dpo(RUN)

### Eval S4

In [ ]:
capcom_eval.eval_stage(RUN, 'S4')

## S5 — GRPO (optional)
`dapo` loss, verifiable reward components normalised per group (`normalize_then_sum`), β 0.02, lr 1e-6, 200 steps. Off on T4.

In [ ]:
if STAGES['S5']: grpo_stage.grpo(RUN)

### Eval S5

In [ ]:
capcom_eval.eval_stage(RUN, 'S5')

## S6 — pick the best stage
The best eval score wins; a 0.9 × best + 0.1 × S3 merge is kept only if it scores higher.

In [ ]:
if STAGES['S6']: merge_stage.select(RUN)
print('final model:', RUN.final())

## S7 — web export
ONNX Runtime GenAI builder int4 (block 32): q4f16 for WebGPU with shader-f16 + q4 for WebGPU without it (the 350M is impractical on WASM: the page falls back to retrieval-only cards), ctx 4096, transformers.js IO, int4 accuracy level `EXPORT_ACC`; greedy parity vs PyTorch; zipped to Drive.

In [ ]:
if STAGES['S7']: print(json.dumps(web_stage.web(RUN, dtypes=('q4',) if SMOKE else ('q4f16', 'q4'), acc=EXPORT_ACC), indent=1))

## Results

In [ ]:
# ---- results ----
print(capcom_eval.table(RUN.evals()))
print('\nfinal:', RUN.final())
d = RUN.done('S7')
if d: print('web model zip:', d['zip'], f"({d['zip_mb']} MB)", '| sizes MB:', d['sizes_mb'], '| parity:', d['parity'])
print('run folder on Drive:', RUN.drive)

## Optional: Hugging Face push

In [ ]:
# ---- optional: push the web model folder to a private Hugging Face repo (Colab secret HF_TOKEN with write access) ----
if HF_PUSH and not LOCAL and RUN.done('S7'):
    from google.colab import userdata
    print(web_stage.push(RUN, HF_REPO, userdata.get('HF_TOKEN')))
else: print('HF push skipped')